# Evaluation Report — Multi-Model Support

**How to run:**
1. Set `CACHE_FILES` below — one entry per model run.
2. `Runtime → Run all`.

In [ ]:
!pip install Levenshtein==0.26.1 plotnine==0.14.5 evaluate==0.4.4 cer==1.2.0 rouge_score==0.1.2 seaborn bitsandbytes python-dateutil --quiet
!pip install hf_transfer --quiet

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1' 
os.environ['USE_HF_TRANSFER'] = '1'
os.environ['USE_HF'] = '1'          
os.environ['MODELSCOPE_CACHE'] = ''


## 1 · Configuration

In [ ]:
import os, json, re
from pathlib import Path


BASE_DIR = Path(os.path.abspath("."))

CACHE_FILES = [
    (BASE_DIR / "rows_cache.json", "auto"),
]

# read adapter_config --> there are model infor
def get_base_model(checkpoint_path: str) -> str:
    cfg = Path(checkpoint_path) / "adapter_config.json"
    if cfg.exists():
        return json.load(open(cfg)).get("base_model_name_or_path", "unknown")
    return "unknown"

def is_valid_checkpoint(ckpt_path):
    cfg = ckpt_path / "adapter_config.json"
    if not cfg.exists(): return False
    try: data = json.load(open(cfg))
    except Exception: return False
    base = data.get("base_model_name_or_path", "")
    return "Qwen2-VL" in base or "Qwen2.5-VL" in base

# get new version
def ckpt_sort_key(p):
    v = re.search(r'[/\\]v(\d+)-', str(p))
    s = re.search(r'checkpoint-(\d+)$', str(p))
    return (int(v.group(1)) if v else 0, int(s.group(1)) if s else 0)

valid_checkpoints = sorted(
    [p for p in (BASE_DIR / 'models/finetune').glob('*/checkpoint-*')
     if is_valid_checkpoint(p)],
    key=ckpt_sort_key
)

if not valid_checkpoints:
    raise FileNotFoundError(f"Not finded checkpoint: {BASE_DIR / 'models/finetune'}")

CHECKPOINT  = str(valid_checkpoints[-1])
DATASET_DIR = BASE_DIR / 'data' / 'swift_dataset'
IMAGES_DIR  = DATASET_DIR / 'images'
MODEL_NAME  = Path(CHECKPOINT).parent.name
CACHE_FILE  = BASE_DIR / 'rows_cache.json'

print(f"BASE_DIR    : {BASE_DIR}")
print(f"CHECKPOINT  : {CHECKPOINT}")
print(f"MODEL_NAME  : {MODEL_NAME}")
print(f"CACHE_FILES : {[(str(c), l) for c, l in CACHE_FILES]}")

print("\nAll valid checkpoints (best last):")
for p in valid_checkpoints:
    marker = "  ← SELECTED" if str(p) == CHECKPOINT else ""
    print(f"  {p.parent.name}/{p.name}  key={ckpt_sort_key(p)}{marker}")

## 2 · Clear VRAM

In [ ]:
import torch, gc

for var in ["model", "trainer", "engine", "optimizer"]:
    if var in globals(): del globals()[var]
gc.collect()
torch.cuda.empty_cache()

free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f"VRAM free: {free:.1f} GB / {total:.1f} GB")
if free < 8:
    print("WARNING: < 8 GB free — inference may OOM.")


## 3 · Shared Helpers

In [ ]:
import numpy as np, pandas as pd
import re
from dateutil import parser as dtparser
from dateutil.parser import ParserError
from datetime import datetime

def _safe(v):
    if v is None: return ""
    if isinstance(v, list): return json.dumps(v, ensure_ascii=False)
    return str(v).strip()

def flatten_dict(d, parent_key="", sep="."):
    items = {}
    for k, v in d.items():
        nk = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):   items.update(flatten_dict(v, nk, sep))
        elif isinstance(v, list): items[nk] = json.dumps(v, ensure_ascii=False)
        else:                     items[nk] = v
    return items


DATE_KEYWORDS_SET    = {"date","ngay","dated","dob","expiry","expire",
                        "issued","issue","valid","from","start","end",
                        "period","birth","day"}
BIRTH_EXCL_PREFIXES  = {"place"}

CATEGORY_MAP = {
    "victoria australia": "VIC",
    "victoria": "VIC",
    "driver licence": "AUS_DRIVER_LICENSE",
    "driving licence": "AUS_DRIVER_LICENSE"
}

def is_date_field(field_name: str) -> bool:
    tokens = set(re.split(r"[_.]", field_name.lower()))
    if "birth" in tokens and tokens & BIRTH_EXCL_PREFIXES:
        tokens.discard("birth")
    return bool(tokens & DATE_KEYWORDS_SET)

def try_parse_date(value: str) -> str:
    if not value or not isinstance(value, str): return value
    v = value.strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", v): return v
    patterns = [
        (r"^(\d{1,2})/(\d{1,2})/(\d{4})$", "%d/%m/%Y"),
        (r"^(\d{4})/(\d{2})/(\d{2})$",      "%Y/%m/%d"),
        (r"^(\d{1,2})-(\d{1,2})-(\d{4})$",  "%d-%m-%Y"),
        (r"^(\d{8})$",                          "%Y%m%d"),
    ]
    for pattern, fmt in patterns:
        if re.fullmatch(pattern, v):
            try: return datetime.strptime(v, fmt).strftime("%Y-%m-%d")
            except ValueError: pass
    try: return dtparser.parse(v, dayfirst=True).strftime("%Y-%m-%d")
    except (ParserError, OverflowError, ValueError): return v

def clean_and_normalize(key: str, value: str) -> str:
    if not isinstance(value, str) or not value:
        return str(value) if value is not None else ""
    
    v = value.strip()
    v_lower = v.lower()

    if v_lower in CATEGORY_MAP:
        return CATEGORY_MAP[v_lower]

    numeric_fields = {"total", "amount", "kwh", "usage", "gst"}
    if any(kw in key.lower() for kw in numeric_fields):
        v = v.replace(",", "")
        v = re.sub(r'[a-zA-Z\s\$]+$', '', v).strip()
        return v

    date_fields = {"date", "dob", "expiry", "issued", "birth"}
    if any(kw in key.lower() for kw in date_fields):
        return v

    return v.upper()

def normalize_all_fields_recursive(record: dict, _pk: str = "") -> dict:
    """Đệ quy chuẩn hóa toàn bộ các trường ngày tháng, số liệu và danh mục."""
    for k, v in record.items():
        if isinstance(v, dict): 
            normalize_all_fields_recursive(v, k)
        elif isinstance(v, str):
            if is_date_field(k):
                record[k] = try_parse_date(v)
            else:
                record[k] = clean_and_normalize(k, v)
    return record

print("Helpers loaded.")

## 4 · Load Caches (Multi-Model)

In [ ]:
from utils.evaluation import find_and_parse_json

all_rows = []

def rewrap_if_needed(pred: dict, gt: dict) -> dict:
    if not isinstance(pred, dict) or not isinstance(gt, dict):
        return pred

    # Lấy các key nested (có value là dict) từ GT làm chuẩn
    gt_nested_keys = {k for k, v in gt.items() if isinstance(v, dict)}
    if not gt_nested_keys:
        return pred  # GT flat → không cần wrap

    # Nếu pred đã có ít nhất 1 key nested giống GT → đúng structure rồi
    if any(k in pred for k in gt_nested_keys):
        return pred

    # pred thiếu các nested key → cần rewrap
    # Phân loại key của pred theo GT nested keys
    result = {}
    for gt_key in gt_nested_keys:
        gt_sub = gt.get(gt_key, {})
        if not isinstance(gt_sub, dict):
            continue
        sub = {k: pred[k] for k in gt_sub if k in pred}
        if sub:
            result[gt_key] = sub

    # Giữ lại các key top-level của GT mà pred có
    gt_top_keys = {k for k, v in gt.items() if not isinstance(v, (dict, list))}
    for k in gt_top_keys:
        if k in pred:
            result[k] = pred[k]

    return result if result else pred

def load_or_infer(cache_path, label):
    cache_path = Path(cache_path)

    # ── cache hit 
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        rows = json.load(open(cache_path))
        for r in rows:
            r["pred"] = rewrap_if_needed(r["pred"], r["gt"])
            r["model_label"] = (
                label if label != "auto"
                else Path(r.get("model", CHECKPOINT)).parent.name
            )
        return rows

    # ── live inference via infer_main ───
    print(f"No cache — running inference for {label}...")
    from swift import infer_main

    test_path   = DATASET_DIR / "conversations_test_swift_format.json"
    result_path = BASE_DIR / "infer_results.jsonl"

    if result_path.exists():
        result_path.unlink()

    argv = [
        "--adapters",       str(CHECKPOINT),
        "--val_dataset",    str(test_path),
        "--result_path",    str(result_path),
        "--max_length",     "8192",
        "--max_pixels",     "8262144",
        "--max_new_tokens", "1024",
        "--temperature",    "0",
        "--infer_backend",  "transformers",
        "--load_data_args", "false",
        "--use_hf",         "true",
        "--logprobs",       "true",
    ]
    infer_main(argv)

    # ── parse results ─────────────────────────────────────────────────────────
    test_data = json.load(open(test_path))

    raw_results = []
    with open(result_path) as f:
        for line in f:
            line = line.strip()
            if line:
                raw_results.append(json.loads(line))

    rows = []
    for i, s in enumerate(test_data):
        gt      = json.loads(s["messages"][2]["content"])
        raw_out = raw_results[i]["response"] if i < len(raw_results) else ""

        pred = find_and_parse_json(raw_out)
        ok   = (
            isinstance(pred, dict)
            and len(pred) > 0
            and any(v is not None and v not in ("...", " ") for v in pred.values())
        )

        rows.append({
            "image":             s["images"][0],
            "doc_type":          gt.get("document_type"),
            "gt":                gt,
            "pred":              {k: v for k, v in pred.items() if not k.startswith("_")} if ok else {},
            "pred_raw":          raw_out if not ok else "",
            "valid_json":        ok,
            "confidence":        0.0,
            "field_confidences": {k: 0.0 for k in gt} if ok else {},
            "model":             CHECKPOINT,
            "model_label":       label if label != "auto" else MODEL_NAME,
        })

    json.dump(rows, open(cache_path, "w"), ensure_ascii=False, indent=2)
    print(f"Cache saved: {cache_path}")
    return rows


# ── main loop ─
for cache_path, label in CACHE_FILES:
    rows = load_or_infer(cache_path, label)
    for r in rows:
        r["gt"] = normalize_all_fields_recursive(r.get("gt", {}))
        if r.get("valid_json") and r.get("pred"):
            r["pred"] = normalize_all_fields_recursive(r.get("pred", {}))
    valid = sum(r["valid_json"] for r in rows)
    print(f"[{rows[0]['model_label'] if rows else label}] {len(rows)} rows | Valid JSON: {valid}/{len(rows)}")
    all_rows.extend(rows)

print(f"\nTotal rows: {len(all_rows)}")


## 5 · Build DataFrame

In [ ]:
import Levenshtein

EXCLUDE_FIELDS = {"back.barcode_number"}

for r in all_rows:
    r["gt"]  = normalize_all_fields_recursive(r.get("gt", {}))
    if r.get("valid_json") and r.get("pred"):
        r["pred"] = normalize_all_fields_recursive(r.get("pred", {}))

for r in all_rows:
    r["gt_flat"]   = flatten_dict(r.get("gt", {}))
    r["pred_flat"] = flatten_dict(r.get("pred", {})) if r.get("valid_json") else {}


all_fields = sorted({k for r in all_rows for k in r["gt_flat"]} - EXCLUDE_FIELDS)
records    = []

for r in all_rows:
    for field in all_fields:
        if field not in r["gt_flat"]: continue
        gt_val = _safe(r["gt_flat"].get(field))

        if not r.get("valid_json"):
            pred_val, dist = "", -4
        elif field not in r["pred_flat"]:
            pred_val, dist = "", -3
        elif gt_val == "":
            pred_val = _safe(r["pred_flat"].get(field, ""))
            dist = -1
        else:
            pred_val = _safe(r["pred_flat"].get(field, ""))
            dist = -2 if pred_val in ("None", "") else Levenshtein.distance(pred_val, gt_val)

        records.append({
            "image"       : r.get("image", ""),
            "doc_type"    : r.get("doc_type", ""),
            "entity"      : field,
            "label_val"   : gt_val,
            "response_val": pred_val,
            "dist"        : dist,
            "valid_json"  : r.get("valid_json", False),
            "model"       : r.get("model", CHECKPOINT),
            "pretty_name" : r.get("model_label", MODEL_NAME),
            "confidence"  : r.get("field_confidences", {}).get(field),
        })

df_multi = pd.DataFrame(records)
print(f"df_multi shape: {df_multi.shape}")
print(f"Models: {df_multi['pretty_name'].unique().tolist()}")

# Date fields detected
date_fields = sorted({f for r in all_rows for f in r["gt_flat"] if is_date_field(f)})
print(f"Date fields detected: {date_fields}")

# Confidence summary
conf_rows = [{"doc_type": r["doc_type"], "confidence": r.get("confidence")}
             for r in all_rows if r.get("confidence") is not None]
if conf_rows:
    print("\nAverage confidence per doc_type:")
    print(pd.DataFrame(conf_rows).groupby("doc_type")["confidence"].mean().round(4))

missing_gt = [f for f in all_fields if 
    (df_multi[df_multi["entity"]==f]["label_val"] == "").mean() > 0.9]
print(f"\nmissing_ground_truth ({len(missing_gt)}): {missing_gt}")


## 6 · Feature Categorization

In [ ]:
from utils.entities import analyze_string_distribution_by_entity, categorize_features

first_model   = df_multi["pretty_name"].unique()[0]
df_single_cat = df_multi[df_multi["pretty_name"] == first_model]

results_df = analyze_string_distribution_by_entity(df_single_cat, col_value="label_val")
feature_categories_enum = categorize_features(results_df, null_percentage_threshold=70, length_mean_threshold=50)

feature_categories = {
    k: (v.value if hasattr(v, 'value') else v)
    for k, v in feature_categories_enum.items()
}

for cat in ["MISSING_GROUND_TRUTH", "SHORT_TEXT", "LONG_TEXT"]:
    members = [e for e, c in feature_categories.items() if c == cat]
    print(f"\n{cat} ({len(members)}): {members}")

### 6.1 · Null Percentage by Entity

In [ ]:
from plotnine import *
null_df = (
    df_single_cat.groupby("entity")["label_val"]
    .apply(lambda x: (x == "").sum() / len(x) * 100)
    .reset_index().rename(columns={"label_val": "metric_value"})
    .sort_values("metric_value", ascending=True)
)
null_df["entity"] = pd.Categorical(null_df["entity"], categories=null_df["entity"].tolist(), ordered=True)

(ggplot(null_df, aes(x="entity", y="metric_value")) +
 geom_bar(stat="identity", fill="steelblue") +
 coord_flip() +
 labs(title="Null Percentage by Entity", x="Entity", y="Null Percentage (%)") +
 theme_bw())


## 7 · Edit Distance Heatmap

In [ ]:
from plotnine import *

df = df_multi.copy()
def encode_dist(x):
    if x in (-2, -3, -4): return {-2:7, -3:8, -4:9}[x]
    if x == -1: return -1
    return min(x, 6)

df["dist_cut"] = df["dist"].apply(encode_dist)

mapper = {
    -1: "missing groundtruth",
     0: "exact match",
     1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6+",
     7: "predicted 'None'",
     8: "key missing",
     9: "invalid JSON"
}
df["dist_cut"]     = pd.Categorical(df["dist_cut"].replace(mapper),
                                    categories=list(mapper.values()), ordered=True)
df["entity_type"]  = df["entity"].map(feature_categories)
df["entity_label"] = df["doc_type"].fillna("UNKNOWN") + " | " + df["entity"]
df_sorted          = df.sort_values(["doc_type", "entity_type", "entity"])

df_entities    = df_sorted[["entity_label","entity_type","doc_type"]].drop_duplicates(subset=["entity_label"]).reset_index(drop=True)
ordered_values = list(df_entities["entity_label"])

change_indices = df_entities.index[
    (df_entities["entity_type"] != df_entities["entity_type"].shift()) |
    (df_entities["doc_type"]    != df_entities["doc_type"].shift())
].tolist()
feature_class_lines = [i + 0.5 for i in change_indices[1:]]

df_sorted["pretty_name_ordered"] = pd.Categorical(
    df_sorted["pretty_name"], categories=sorted(df_sorted["pretty_name"].unique()), ordered=True
)

custom_colors = ["#bababa","#66c2a5","#abdda4","#e6f598","#ffffbf",
                 "#fee08b","#fdae61","#f46d43","#abd9e9","#74add1","#bebada"]

plot = (
    ggplot(df_sorted, aes(x="entity_label", fill="factor(dist_cut)"))
    + geom_bar(position="stack", color="black")
    + facet_wrap("~pretty_name_ordered", scales="free")
    + labs(x="Doc Type | Field", y="Count", fill="Char. edit distance")
    + coord_flip()
    + scale_fill_manual(values=custom_colors, labels=list(mapper.values()))
    + theme_minimal()
    + theme(figure_size=(16, 14))
    + scale_x_discrete(limits=ordered_values)
)
from IPython.display import Markdown, display
display(Markdown("### All Entities"))
plot.show()


## 8 · Model Performance Metrics

In [ ]:
import evaluate
from cer import calculate_cer   # cer==1.2.0 → exports calculate_cer
import evaluate
from cer import calculate_cer

_hf_metrics = {name: evaluate.load(name) for name in ["exact_match", "bleu", "rouge"]}

def calculate_metrics(references, predictions):
    """
    Returns pd.Series with: exact_match, cer_score, bleu,
    rouge1, rouge2, rougeL, rougeLsum.
    """
    pairs = [(r, p) for r, p in zip(references, predictions) if str(r).strip() != ""]
    if not pairs:
        return pd.Series({"exact_match": 0.0, "cer_score": 0.0,
                          "bleu": 0.0, "rouge1": 0.0, "rouge2": 0.0,
                          "rougeL": 0.0, "rougeLsum": 0.0})
    refs, preds = zip(*pairs)
    refs  = [str(r) for r in refs]
    preds = [str(p) for p in preds]

    results = {}
    for name, metric in _hf_metrics.items():
        try:
            results.update(metric.compute(predictions=preds, references=refs) or {})
        except Exception:
            # Gán giá trị 0.0 nếu BLEU hoặc ROUGE gặp lỗi mảng rỗng
            results[name] = 0.0

    try:
        results["cer_score"] = calculate_cer(refs, preds)
    except Exception:
        results["cer_score"] = 0.0

    return pd.Series(results)

print("Metrics ready.")
print("  HuggingFace metrics:", list(_hf_metrics.keys()))
print("  CER: via cer.calculate_cer")

In [ ]:


relevant_entities = [e for e, c in feature_categories.items()
                     if c != "MISSING_GROUND_TRUTH"]
df_filtered = df_multi[df_multi["entity"].isin(relevant_entities)].copy()

metrics_records = []
for keys, group in df_filtered.groupby(["model", "pretty_name", "entity"]):
    res = calculate_metrics(
        group["label_val"].astype(str).tolist(),
        group["response_val"].astype(str).tolist()
    )
    row = res.to_dict()
    row["model"] = keys[0]
    row["pretty_name"] = keys[1]
    row["entity"] = keys[2]
    metrics_records.append(row)

df_eval = pd.DataFrame(metrics_records)

print(f"df_eval shape: {df_eval.shape}")
print(f"Columns: {df_eval.columns.tolist()}")
df_eval.head(3)

### 8.1 · Exact Match per Entity

In [ ]:
from plotnine import *

# Build ordered entity list (exclude missing GT)
df_ent_rel = df_entities[df_entities["entity_label"].apply(
    lambda x: x.split(" | ")[-1] if " | " in x else x
).isin(relevant_entities)].reset_index(drop=True)
ordered_relevant = list(df_ent_rel["entity_label"])

# Map entity → entity_label for plotting
entity_to_label = {
    row.split(" | ")[-1]: row
    for row in ordered_relevant
}
df_eval["entity_label"] = df_eval["entity"].map(
    lambda e: entity_to_label.get(e, e)
)

change_idx_rel = df_ent_rel.index[
    df_ent_rel["entity_type"] != df_ent_rel["entity_type"].shift()
].tolist()
feat_lines_rel = [i + 0.5 for i in change_idx_rel[1:]]

(
    ggplot(df_eval, aes(x="entity_label", y="exact_match", fill="pretty_name"))
    + geom_bar(stat="identity", position="dodge", colour="gray")
    + labs(title="Exact Match per Entity (higher = better)",
           x="Entity", y="Exact Match", fill="Model")
    + coord_flip()
    + scale_fill_brewer(type="qual", palette="Set3")
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + scale_y_continuous(breaks=[x/100 for x in range(0,101,10)],
                         labels=[f"{i}%" for i in range(0,101,10)])
    + geom_hline(yintercept=[x/100 for x in range(0,101,10)],
                 color="darkgray", size=0.4, alpha=0.5)
    + geom_vline(xintercept=feat_lines_rel, linetype="dashed",
                 color="#4d4d4d", size=2.0)
    + scale_x_discrete(limits=ordered_relevant)
)


### 8.2 · Character Error Rate (CER) per Entity

In [ ]:
(
    ggplot(df_eval, aes(x="entity_label", y="cer_score", fill="pretty_name"))
    + geom_bar(stat="identity", position="dodge", color="gray")
    + labs(title="CER per Entity (lower = better)", x="Entity", y="CER", fill="Model")
    + coord_flip()
    + scale_fill_brewer(type="qual", palette="Set3")
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + geom_vline(xintercept=feat_lines_rel, linetype="dashed",
                 color="#4d4d4d", size=2.0)
    + scale_x_discrete(limits=ordered_relevant)
)


### 8.3 · Aggregate Metrics Table

- **Short text** → Exact Match + CER  
- **Long text** → ROUGE scores

In [ ]:
import numpy as np

short_ents = [e for e, c in feature_categories.items() if c == "SHORT_TEXT"]
long_ents  = [e for e, c in feature_categories.items() if c ==  "LONG_TEXT"]

exact_agg = (
    df_eval[df_eval["entity"].isin(short_ents)]
    .groupby(["model", "pretty_name"])
    .agg({"exact_match": "mean", "cer_score": "mean"})
    .reset_index()
    .rename(columns={"exact_match": "accuracy (exact match)"})
)

# ROUGE only if long_text entities exist
if long_ents:
    rouge_agg = (
        df_eval[df_eval["entity"].isin(long_ents)]
        .groupby(["model", "pretty_name"])
        .agg({"rouge1": "mean", "rouge2": "mean", "rougeL": "mean", "rougeLsum": "mean"})
        .reset_index()
    )
    aggregated_metrics = rouge_agg.merge(exact_agg, on=["model", "pretty_name"], how="inner")
    metric_cols = ["rouge1", "rouge2", "rougeL", "rougeLsum", "accuracy (exact match)", "cer_score"]
else:
    aggregated_metrics = exact_agg.copy()
    metric_cols = ["accuracy (exact match)", "cer_score"]

aggregated_metrics[metric_cols] = aggregated_metrics[metric_cols].round(3)

def highlight_max(s, props=""): return np.where(s == np.nanmax(s.values), props, "")
def highlight_min(s, props=""): return np.where(s == np.nanmin(s.values), props, "")

hi_cols  = [c for c in metric_cols if c != "cer_score"]
lo_cols  = ["cer_score"]

display(aggregated_metrics.style
    .apply(highlight_max, props="background-color:#99d594;", axis=0, subset=hi_cols)
    .apply(highlight_min, props="background-color:#99d594;", axis=0, subset=lo_cols)
)


## 9 · Visual Diff — Prediction vs Ground Truth

In [ ]:
# from utils.docdiff import get_diff, image_formatter
# from IPython.display import HTML, display
# import pandas as pd

# N_SHOW = 10
# first_label = df_multi["pretty_name"].unique()[0]
# sample_rows = [r for r in all_rows if r.get("model_label") == first_label][:N_SHOW]

# # Chuyển đổi định dạng để tương thích với utils.docdiff
# df_sample = pd.DataFrame(sample_rows)
# df_sample = df_sample.rename(columns={"gt": "labels", "pred": "response"})

# for i, row in df_sample.iterrows():
#     p = Path(row.get("image", ""))
#     if not p.is_absolute(): p = IMAGES_DIR / p.name
    
#     status = "✓ valid JSON" if row.get("valid_json") else "✗ invalid JSON"
#     conf   = row.get("confidence")
#     status_text = f"{status}  | conf={conf:.3f}" if conf else status
    
#     display(HTML(f"<h3>#{i+1} | {row.get('doc_type','?')} | {row.get('model_label','?')} | {status_text}</h3>"))
    
#     try:
#         display(HTML(image_formatter(str(p))))
#     except Exception as e:
#         print(f"Could not load image: {e}")
        
#     if row.get("valid_json"):
#         diff_html = get_diff(row)
#         display(HTML(diff_html))
#     else:
#         print(f"Raw output:\n{row.get('pred_raw', '')}")

import textwrap
from PIL import Image
import matplotlib.pyplot as plt

N_SHOW = 15
COL_W  = 35  # column width for GT and Pred

def flatten_dict(d, prefix=''):
    out = {}
    for k, v in (d or {}).items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            for i, item in enumerate(v):
                if isinstance(item, dict):
                    out.update(flatten_dict(item, f"{key}[{i}]"))
                else:
                    out[f"{key}[{i}]"] = item
        else:
            out[key] = v
    return out

for i, r in enumerate(rows[:N_SHOW]):
    gt_flat   = flatten_dict(r['gt'])
    pred_flat = flatten_dict(r['pred']) if r['valid_json'] else {}

    p = Path(r['image'])
    if not p.is_absolute(): p = IMAGES_DIR / p.name
    try:
        img = Image.open(p).convert('RGB')
        plt.figure(figsize=(5, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"#{i+1} | {r['doc_type']}")
        plt.show()
    except Exception as e:
        print(f"Could not load image: {e}")

    status = '✓ valid JSON' if r['valid_json'] else '✗ invalid JSON'
    print(status)
    print(f"{'Field':<40} {'GT':<{COL_W}} Pred")
    print("-" * (40 + COL_W * 2 + 2))

    for k in sorted(gt_flat.keys()):
        gv = str(gt_flat.get(k, '')).strip()
        pv = str(pred_flat.get(k, '')).strip()
        ok = '✓' if gv == pv else '✗'

        gv_lines = textwrap.wrap(gv, COL_W) or ['']
        pv_lines = textwrap.wrap(pv, COL_W) or ['']
        n_lines  = max(len(gv_lines), len(pv_lines))

        for j in range(n_lines):
            prefix = f"{ok} {k:<38}" if j == 0 else f"  {'':<38}"
            g  = gv_lines[j] if j < len(gv_lines) else ''
            p_ = pv_lines[j] if j < len(pv_lines) else ''
            print(f"{prefix} {g:<{COL_W}} {p_}")

    print()